# Bronze — Statistics Iceland GDP Ingestion

Fetches quarterly GDP year-on-year growth from Statistics Iceland (Hagstofa Íslands) via the PX-Web REST API.

**Source:** Hagstofa Íslands PX-Web API  
**Dataset:** THJ01601 — Quarterly GDP (Mælikvarði=2 YoY growth, Skipting=14 Total GDP)  
**Output:** `bronze.statistics_iceland_gdp` (Delta table)  
**Schedule:** Quarterly  

In [ ]:
import requests
import json
import pandas as pd

API_URL = (
    "https://px.hagstofa.is/pxis/api/v1/is"
    "/Efnahagur/thjodhagsreikningar/landsframl"
    "/2_landsframleidsla_arsfj/THJ01601.px"
)
START_YEAR = 2022

payload = {
    "query": [
        {"code": "Mælikvarði", "selection": {"filter": "item", "values": ["2"]}},
        {"code": "Skipting",   "selection": {"filter": "item", "values": ["14"]}},
    ],
    "response": {"format": "json"},
}

response = requests.post(API_URL, json=payload)
response.raise_for_status()
data = response.json()

print(f"Total data points returned: {len(data['data'])}")

In [ ]:
records = []
for item in data["data"]:
    quarter = item["key"][2]
    raw_value = item["values"][0]
    records.append({
        "quarter": quarter,
        "year": int(quarter[:4]),
        "q": int(quarter[5]),
        "metric": "gdp_yoy_growth",
        "value": float(raw_value) if raw_value != ".." else None,
        "ingested_at": pd.Timestamp.now(),
    })

df_gdp = pd.DataFrame(records)
df_gdp = df_gdp[df_gdp["year"] >= START_YEAR]

print(f"Filtered rows (>= {START_YEAR}): {len(df_gdp)}")
print(df_gdp.to_string(index=False))

if len(df_gdp) == 0:
    raise ValueError(f"[bronze_statistics] No GDP rows for year >= {START_YEAR} - halting.")

In [ ]:
spark_df = spark.createDataFrame(df_gdp)

spark_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze.statistics_iceland_gdp")

print(f"Saved to bronze.statistics_iceland_gdp — {spark_df.count()} rows")